# Threshold Selection for High-Stakes Decisions

Your model outputs probabilities. You need to make binary decisions.

**The naive approach**: threshold at 0.5  
**The reality**: The right threshold depends entirely on your use case.

In cancer screening:
- **Too aggressive** (low threshold): Many false positives → unnecessary biopsies
- **Too conservative** (high threshold): Missed cancers → delayed treatment

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, confusion_matrix

import sys
sys.path.append('..')
from evaluation import ThresholdOptimizer

sns.set_style("whitegrid")
np.random.seed(42)

## The Scenario

We have a breast cancer screening model with ~5% prevalence.

In [ ]:
X, y = make_classification(
    n_samples=10000,
    n_features=20,
    n_informative=10,
    weights=[0.95, 0.05],
    random_state=42,
    flip_y=0.05
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"Positive rate: {y_test.mean():.1%}")

In [ ]:
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train, y_train)
y_prob = model.predict_proba(X_test)[:, 1]

print(f"Probability range: [{y_prob.min():.3f}, {y_prob.max():.3f}]")

## The Default: Threshold = 0.5

In [ ]:
y_pred_default = (y_prob >= 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_default).ravel()

sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)

print("Default Threshold = 0.5")
print("=" * 40)
print(f"True Positives:  {tp}")
print(f"False Negatives: {fn}  <- MISSED CANCERS")
print(f"False Positives: {fp}")
print(f"True Negatives:  {tn}")
print()
print(f"Sensitivity: {sensitivity:.1%}")
print(f"Specificity: {specificity:.1%}")
print()
print(f"We're missing {fn} cancers out of {tp + fn} total!")

## Finding Better Thresholds

In [ ]:
optimizer = ThresholdOptimizer()

results = {
    'Default (0.5)': 0.5,
    'Youden J': optimizer.optimize_youden(y_test, y_prob).optimal_threshold,
    'Sensitivity >= 95%': optimizer.optimize_sensitivity(y_test, y_prob, min_sensitivity=0.95).optimal_threshold,
    'Sensitivity >= 90%': optimizer.optimize_sensitivity(y_test, y_prob, min_sensitivity=0.90).optimal_threshold,
    'Max F1': optimizer.optimize_f1(y_test, y_prob).optimal_threshold,
}

for name, thresh in results.items():
    print(f"{name}: {thresh:.3f}")

## Comparing Operating Points

In [ ]:
comparison = []

for name, thresh in results.items():
    y_pred = (y_prob >= thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    
    comparison.append({
        'Strategy': name,
        'Threshold': f"{thresh:.3f}",
        'Sensitivity': f"{tp / (tp + fn):.1%}",
        'Specificity': f"{tn / (tn + fp):.1%}",
        'Missed Cancers': fn,
        'Unnecessary Biopsies': fp
    })

comparison_df = pd.DataFrame(comparison)
print(comparison_df.to_string(index=False))

## Visualizing the Trade-off

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

fig, ax = plt.subplots(figsize=(10, 8))

ax.plot(fpr, tpr, 'b-', linewidth=2, label='Model')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')

colors = plt.cm.Set1(np.linspace(0, 1, len(results)))
for (name, thresh), color in zip(results.items(), colors):
    idx = np.argmin(np.abs(thresholds - thresh))
    ax.scatter(fpr[idx], tpr[idx], s=150, c=[color], marker='o', 
               edgecolors='black', linewidth=2, zorder=5)
    ax.annotate(f"{name}\n(t={thresh:.2f})", 
                xy=(fpr[idx], tpr[idx]),
                xytext=(fpr[idx] + 0.05, tpr[idx] - 0.05),
                fontsize=9)

ax.set_xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
ax.set_ylabel('True Positive Rate (Sensitivity)', fontsize=12)
ax.set_title('ROC Curve with Operating Points', fontsize=14)
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## The Bottom Line

**For cancer screening**, we'd likely choose **Sensitivity >= 95%** as our operating point.

In [ ]:
recommended = optimizer.optimize_sensitivity(y_test, y_prob, min_sensitivity=0.95)

print("\n" + "=" * 60)
print(" RECOMMENDED OPERATING POINT FOR CANCER SCREENING")
print("=" * 60)
print(recommended)